In [1]:
# Instalação de dependências 
# !pip install mrmr-selection scikit-learn pandas numpy matplotlib scipy tabulate openpyxl

import numpy as np
import pandas as pd

from sklearn.svm import SVC
from sklearn.model_selection import RepeatedStratifiedKFold, RepeatedKFold, cross_val_score
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from math import erf

In [7]:
def mrmr_filter(X, y, k=100, random_state=None):
    """
    Filtro MRMR simplificado:
    - Relevância: mutual_info_classif(X_j, y)
    - Redundância: média da |correlação de Pearson| com as features já escolhidas.

    X: array (n_amostras, n_features)
    y: vetor de rótulos
    k: número de atributos a selecionar
    """
    rng = np.random.RandomState(random_state)

    # Garante array numpy
    if isinstance(X, pd.DataFrame):
        X_np = X.values
    else:
        X_np = np.asarray(X)

    y_arr = np.asarray(y)

    # y para MI precisa ser discreto
    le = LabelEncoder()
    y_enc = le.fit_transform(y_arr)

    n_samples, n_feats = X_np.shape
    k = min(k, n_feats)

    # relevância
    mi = mutual_info_classif(X_np, y_enc, discrete_features=False,
                             random_state=random_state)

    # 1ª feature: maior MI com a classe
    selected = [int(np.argmax(mi))]
    candidatos = set(range(n_feats)) - set(selected)

    # função de redundância = média da |correlação|
    def redundancia(idx, selected_idxs):
        vals = []
        xj = X_np[:, idx]
        for s in selected_idxs:
            xs = X_np[:, s]
            corr = np.corrcoef(xj, xs)[0, 1]
            if np.isnan(corr):
                corr = 0.0
            vals.append(abs(corr))
        if len(vals) == 0:
            return 0.0
        return float(np.mean(vals))

    # seleção gulosa MRMR
    while len(selected) < k and len(candidatos) > 0:
        best_score = -np.inf
        best_feat = None
        for j in candidatos:
            rel = mi[j]
            red = redundancia(j, selected)
            score = rel - red  # MRMR
            if score > best_score:
                best_score = score
                best_feat = j

        selected.append(best_feat)
        candidatos.remove(best_feat)

    return np.array(selected, dtype=int)


In [ ]:
class BHOASelecaoDeAtributos:
    """
    BHOA com:
      - SVM (kernel linear) + 10-fold CV (repetível)
      - Penalização pelo nº de atributos (α, β)
      - X-Shaped Transfer Function + crossover (opcional)
      - Filtro MRMR antes do BHOA (opcional)
    """

    def __init__(
        self,
        n_cavalos=35,
        max_iteracoes=60,
        alfa=0.99,
        beta=0.01,
        usar_xshape=True,
        usar_mrmr=True,
        n_mrmr_features=100,
        cv_splits=10,
        cv_repeats=1,
        random_state=42,
    ):
        print('teste')
        self.n_cavalos = n_cavalos
        self.max_iteracoes = max_iteracoes
        self.alfa = alfa
        self.beta = beta
        self.usar_xshape = usar_xshape
        self.usar_mrmr = usar_mrmr
        self.n_mrmr_features = n_mrmr_features
        self.cv_splits = cv_splits
        self.cv_repeats = cv_repeats
        self.random_state = random_state

        # Atributos preenchidos após executar()
        self.melhor_solucao = None          # máscara binária nas features do espaço reduzido
        self.melhor_Fitness = -np.inf
        self.historico_Fitness = []
        self.indices_mrmr_ = None           # índices das features após MRMR em relação ao X original
        self.indices_selecionados_ = None   # índices finais em relação ao X original

    def _prepara_dados(self, X, y):
        """Converte para numpy e aplica MRMR se habilitado."""
        if isinstance(X, pd.DataFrame):
            X_np = X.values
        else:
            X_np = np.asarray(X)

        y_arr = np.asarray(y)

        # Tratar valores ausentes: imputação por média das colunas
        if np.isnan(X_np).any():
            imputer = SimpleImputer(strategy="mean")
            X_np = imputer.fit_transform(X_np)

        if self.usar_mrmr:
            idx = mrmr_filter(
                X_np, y_arr,
                k=self.n_mrmr_features,
                random_state=self.random_state
            )
            self.indices_mrmr_ = idx
            X_work = X_np[:, idx]
        else:
            self.indices_mrmr_ = np.arange(X_np.shape[1])
            X_work = X_np

        self.n_feats_trabalho_ = X_work.shape[1]

        # CV fixo/repetível
        # Ajusta n_splits se alguma classe tiver menos amostras que self.cv_splits
        unique, counts = np.unique(y_arr, return_counts=True)
        min_count = int(counts.min()) if counts.size > 0 else 0

        if min_count >= 2:
            n_splits = min(self.cv_splits, min_count)
            if n_splits < 2:
                n_splits = 2
            if n_splits != self.cv_splits:
                print(f"Ajustando cv n_splits de {self.cv_splits} para {n_splits} (classe mínima={min_count})")
            self.cv_ = RepeatedStratifiedKFold(
                n_splits=n_splits,
                n_repeats=self.cv_repeats,
                random_state=self.random_state,
            )
        else:
            # Há pelo menos uma classe com <2 amostras — não é possível usar Stratified folds.
            # Usamos RepeatedKFold (não estratificado) como fallback com 2 folds.
            print(f"Fallback para RepeatedKFold: pelo menos uma classe tem <2 amostras (min_count={min_count})")
            self.cv_ = RepeatedKFold(
                n_splits=2,
                n_repeats=self.cv_repeats,
                random_state=self.random_state,
            )

        return X_work, y_arr

    def _calcular_Fitness(self, mascara, X, y):
        """
        mascara: vetor binário (0/1) de tamanho n_feats_trabalho_
        X: dados já no espaço reduzido (após MRMR)
        y: rótulos
        """
        cols = np.where(mascara == 1)[0]
        if cols.size == 0:
            return 0.0

        X_red = X[:, cols]

        clf = SVC(kernel='linear')
        scores = cross_val_score(clf, X_red, y, cv=self.cv_)
        acc = scores.mean()

        N = self.n_feats_trabalho_
        S = cols.size
        fitness = self.alfa * acc + self.beta * (1.0 - (S / N))
        return float(fitness)

    def _xshape_update(self, pos_i, vel_i, fit_i, X, y, rng):
        """
        Implementa a X-Shaped Transfer Function (inspirada no Algoritmo 2 do artigo).
        - Gera D e G a partir de vel_i
        - Escolhe Z = melhor(D, G)
        - Se Z melhora pos_i → substitui
        - Caso contrário → crossover entre Z e pos_i e pega o melhor filho (se melhorar)
        """
        v = vel_i

        # Sigmóides W1 e W2
        W1 = 1.0 / (1.0 + np.exp(-v))
        W2 = 1.0 / (1.0 + np.exp(v))

        # Gera D e G
        rand1 = rng.rand(self.n_feats_trabalho_)
        rand2 = rng.rand(self.n_feats_trabalho_)

        D = (rand1 < W1).astype(int)
        G = (rand2 > W2).astype(int)

        # Garante pelo menos 1 atributo em cada
        if D.sum() == 0:
            D[rng.randint(self.n_feats_trabalho_)] = 1
        if G.sum() == 0:
            G[rng.randint(self.n_feats_trabalho_)] = 1

        fit_D = self._calcular_Fitness(D, X, y)
        fit_G = self._calcular_Fitness(G, X, y)

        if fit_D >= fit_G:
            Z = D
            fit_Z = fit_D
        else:
            Z = G
            fit_Z = fit_G

        # Se Z já melhora o cavalo atual, usa Z
        if fit_Z >= fit_i:
            return Z, fit_Z

        # Senão, faz crossover entre Z e pos_i
        mask = rng.rand(self.n_feats_trabalho_) < 0.5
        filho1 = np.where(mask, Z, pos_i)
        filho2 = np.where(mask, pos_i, Z)

        if filho1.sum() == 0:
            filho1[rng.randint(self.n_feats_trabalho_)] = 1
        if filho2.sum() == 0:
            filho2[rng.randint(self.n_feats_trabalho_)] = 1

        fit_f1 = self._calcular_Fitness(filho1, X, y)
        fit_f2 = self._calcular_Fitness(filho2, X, y)

        best_mask = pos_i
        best_fit = fit_i

        if fit_f1 >= best_fit:
            best_mask = filho1
            best_fit = fit_f1
        if fit_f2 >= best_fit:
            best_mask = filho2
            best_fit = fit_f2

        return best_mask, best_fit

    def executar(self, X, y):
        """
        Roda o MRMR-BHOA (com ou sem X-Shaped, com ou sem MRMR) e
        devolve:
            - melhor_solucao: máscara binária nas features do espaço reduzido
            - melhor_Fitness: valor da função objetivo

        As features selecionadas em relação ao X original ficam em:
            self.indices_selecionados_
        """
        rng = np.random.RandomState(self.random_state)

        X_work, y_arr = self._prepara_dados(X, y)
        n_feats = self.n_feats_trabalho_

        # Inicialização da população
        pos = rng.randint(0, 2, size=(self.n_cavalos, n_feats))
        # garante pelo menos 1 atributo selecionado por cavalo
        for i in range(self.n_cavalos):
            if pos[i].sum() == 0:
                pos[i, rng.randint(n_feats)] = 1

        vel = np.zeros((self.n_cavalos, n_feats), dtype=float)
        fit = np.zeros(self.n_cavalos, dtype=float)

        # Fitness inicial
        for i in range(self.n_cavalos):
            fit[i] = self._calcular_Fitness(pos[i], X_work, y_arr)

        # Melhor global
        best_idx = int(np.argmax(fit))
        self.melhor_Fitness = fit[best_idx]
        self.melhor_solucao = pos[best_idx].copy()
        self.historico_Fitness = [self.melhor_Fitness]

        # Loop principal do BHOA
        for t in range(self.max_iteracoes):
            # Ordena por fitness (decrescente)
            ordem = np.argsort(fit)[::-1]
            pos = pos[ordem]
            vel = vel[ordem]
            fit = fit[ordem]

            # Atualiza melhor global
            if fit[0] > self.melhor_Fitness:
                self.melhor_Fitness = fit[0]
                self.melhor_solucao = pos[0].copy()
            self.historico_Fitness.append(self.melhor_Fitness)

            # Define grupos α, β, γ, δ
            n_alfa = max(1, int(0.10 * self.n_cavalos))
            n_beta = max(1, int(0.20 * self.n_cavalos))
            n_gama = max(1, int(0.30 * self.n_cavalos))
            # δ = restante

            p_best = pos[0].astype(float)
            p_mean = pos.mean(axis=0)
            p_worst = pos[-1].astype(float)

            novas_pos = np.zeros_like(pos)
            novos_fit = np.zeros_like(fit)

            for i in range(self.n_cavalos):
                # Componentes de comportamento (simplificados, mas inspirados no HOA)
                G = rng.uniform(-1, 1, n_feats) * 1.5
                H = 1.5 * (p_best - pos[i])
                S = 1.5 * (p_mean - pos[i])
                I = rng.rand() * (p_best - pos[i])
                D = -1.0 * (p_worst - pos[i])
                R = rng.uniform(-1, 1, n_feats)

                if i < n_alfa:
                    vel[i] = 0.9 * vel[i] + G + D
                elif i < n_alfa + n_beta:
                    vel[i] = 0.9 * vel[i] + G + H + S + D
                elif i < n_alfa + n_beta + n_gama:
                    vel[i] = 0.9 * vel[i] + G + H + S + I + D + R
                else:
                    vel[i] = 0.9 * vel[i] + G + I + R

                if self.usar_xshape:
                    nova_pos_i, novo_fit_i = self._xshape_update(
                        pos[i], vel[i], fit[i],
                        X_work, y_arr, rng
                    )
                else:
                    # Versão simples S-shaped (logística)
                    prob = 1.0 / (1.0 + np.exp(-vel[i]))
                    nova_pos_i = (rng.rand(n_feats) < prob).astype(int)
                    if nova_pos_i.sum() == 0:
                        nova_pos_i[rng.randint(n_feats)] = 1
                    novo_fit_i = self._calcular_Fitness(nova_pos_i, X_work, y_arr)

                novas_pos[i] = nova_pos_i
                novos_fit[i] = novo_fit_i

            pos = novas_pos
            fit = novos_fit

        # Mapeia seleção final para os índices originais de X
        mask_final = self.melhor_solucao.astype(bool)
        self.indices_selecionados_ = self.indices_mrmr_[mask_final]

        return self.melhor_solucao, self.melhor_Fitness

In [ ]:
# -----------------------
#  TRANSFER FUNCTIONS S e V
# -----------------------
def s_tf(v, variant='s1'):
    """
    S-shaped transfer functions S1..S4 (vetorizado).
    v: array-like
    variant: 's1','s2','s3','s4'
    Retorna: T(v) in [0,1]
    """
    v = np.asarray(v, dtype=float)
    name = variant.lower()
    if name == 's1':
        return 1.0 / (1.0 + np.exp(-2.0 * v))
    elif name == 's2':
        return 1.0 / (1.0 + np.exp(-1.0 * v))
    elif name == 's3':
        return 1.0 / (1.0 + np.exp(-(v / 2.0)))
    elif name == 's4':
        return 1.0 / (1.0 + np.exp(-(v / 3.0)))
    else:
        raise ValueError("variant must be 's1','s2','s3','s4'")

def v_tf(v, variant='v1'):
    """
    V-shaped transfer functions V1..V4 (vetorizado).
    v: array-like
    variant: 'v1','v2','v3','v4'
    Retorna: T(v) in [0,1]
    """
    v = np.asarray(v, dtype=float)
    name = variant.lower()
    if name == 'v1':
        # V1: |erf((sqrt(pi)/2) * v)|
        try:
            from scipy.special import erf as sp_erf
            return np.abs(sp_erf((np.sqrt(np.pi) / 2.0) * v))
        except Exception:
            # fallback para math.erf (vetorizado)
            from math import erf as math_erf
            erf_vec = np.vectorize(math_erf)
            return np.abs(erf_vec((np.sqrt(np.pi) / 2.0) * v))
    elif name == 'v2':
        # V2: |tanh(v)|
        return np.abs(np.tanh(v))
    elif name == 'v3':
        # V3: | v / sqrt(1 + v^2) |
        return np.abs(v / np.sqrt(1.0 + v**2))
    elif name == 'v4':
        # V4: | (2/pi) * arctan((2/pi) * v) |
        return np.abs((2.0 / np.pi) * np.arctan((2.0 / np.pi) * v))
    else:
        raise ValueError("variant must be 'v1','v2','v3','v4'")

def s_map_to_binary(v, variant='s1', rng=None):
    """
    Mapeamento S-shaped: bit = 1 se rand < T(v), caso contrário 0.
    rng: np.random.RandomState ou None
    """
    T = s_tf(v, variant)
    if rng is None:
        r = np.random.rand(*T.shape)
    else:
        # RandomState: usar rand(shape)
        r = rng.rand(*T.shape)
    return (r < T).astype(int)

def v_map_to_binary_flip(v, current_bits, variant='v1', rng=None):
    """
    Mapeamento V-shaped: flip do bit atual com probabilidade T(v).
    current_bits: array de 0/1
    rng: np.random.RandomState ou None
    """
    T = v_tf(v, variant)
    if rng is None:
        r = np.random.rand(*T.shape)
    else:
        r = rng.rand(*T.shape)
    flip = (r < T)
    current_bits = np.asarray(current_bits, dtype=int)
    return np.where(flip, 1 - current_bits, current_bits)

In [9]:
# =========================
#  CARREGAR DATASETS
# =========================

# 1) Lymphoma: rótulo na coluna "class"
df_lym = pd.read_excel("Lymphoma.xlsx")
X_lym = df_lym.drop(columns=["class"])
y_lym = df_lym["class"]

# 2) Brain_Tumor1: rótulo na PRIMEIRA coluna, genes nas demais
df_brain = pd.read_excel("Brain_Tumor1.xlsx")
y_brain = df_brain.iloc[:, 0]       # primeira coluna = classe (0,1,2,3,4)
X_brain = df_brain.iloc[:, 1:]      # demais colunas = atributos

# 3) Prostate_Tumor: rótulo na PRIMEIRA coluna, genes nas demais
df_pros = pd.read_excel("Prostate_Tumor.xlsx")
y_pros = df_pros.iloc[:, 0]         # primeira coluna = classe (0/1)
X_pros = df_pros.iloc[:, 1:]        # demais colunas = atributos

In [ ]:
# =========================
#  FUNÇÃO AUXILIAR PRA RODAR O BHOA EM UM DATASET
# =========================

def rodar_bhoa_no_dataset_tabela(nome, X, y):
    """
    Versão da função que executa o BHOA no dataset e imprime UMA TABELA
    com o formato parecido com a imagem (colunas: Accuracy (%)[Best,Mean,Worst,STD],
    Number of Genes[Mean,STD], Running Time (s)).
    Retorna: (bhoa, mask, fit, acc_estimada, df)  -- df é o DataFrame impressO.
    """
    import time
    import numpy as np
    import pandas as pd

    print(f"\n=== Dataset: {nome} ===")
    print("Formato X:", X.shape)
    classes, counts = np.unique(y, return_counts=True)
    print("Classes:", dict(zip(classes, counts)))

    bhoa = BHOASelecaoDeAtributos(
        n_cavalos=35,
        max_iteracoes=60,
        alfa=0.99,
        beta=0.01,
        usar_xshape=True,
        usar_mrmr=True,
        n_mrmr_features=100,
        cv_splits=10,
        cv_repeats=1,
        random_state=42,
    )

    t0 = time.time()
    mask, fit = bhoa.executar(X, y)
    tempo = time.time() - t0

    # número de genes selecionados no espaço de trabalho (após MRMR)
    S = int(mask.sum())
    N = bhoa.n_feats_trabalho_

    # estimar acurácia a partir do fitness:
    # fitness = alfa * acc + beta * (1 - S/N)
    acc_estimada = (fit - bhoa.beta * (1.0 - (S / N))) / bhoa.alfa

    print(f"Melhor fitness: {fit:.4f}")
    print(f"Nº de genes selecionados (após MRMR): {S} de {N}")
    print(f"Acurácia estimada (a partir do fitness): {acc_estimada:.4f}")
    print("Índices das features selecionadas no X original:")
    print(bhoa.indices_selecionados_)

    # montar DataFrame com MultiIndex para ficar parecido com a tabela da imagem
    cols = pd.MultiIndex.from_tuples([
        ("Dataset", ""),
        ("Accuracy (%)", "Best"),
        ("Accuracy (%)", "Mean"),
        ("Accuracy (%)", "Worst"),
        ("Accuracy (%)", "STD"),
        ("Number of Genes", "Mean"),
        ("Number of Genes", "STD"),
        ("Running Time (s)", "")
    ])

    # Nesta versão usamos a acurácia estimada para Best/Mean/Worst (se desejar, pode calcular estatísticas reais
    # a partir de múltiplas execuções ou resultados CV do BHOA, caso disponíveis).
    linha = [
        nome,
        acc_estimada, 
        acc_estimada,
        acc_estimada,
        0.0,          # STD (não calculada aqui -> zero)
        S,
        0.0,          # STD número de genes (não calculada aqui -> zero)
        tempo
    ]

    df = pd.DataFrame([linha], columns=cols)

    # impressão formatada (sem índice, duas casas decimais)
    pd.set_option("display.colheader_justify", "center")
    print("\nResultado resumido:")
    print(df.to_string(index=False, float_format="%.2f"))

    return bhoa, mask, fit, acc_estimada, df

In [17]:
# =========================
#  RODAR NOS 3 DATASETS
# =========================

bhoa_lym, mask_lym, fit_lym, acc_lym, df_lym = rodar_bhoa_no_dataset_tabela("Lymphoma", X_lym, y_lym)
bhoa_brain, mask_brain, fit_brain, acc_brain, df_brain = rodar_bhoa_no_dataset_tabela("Brain_Tumor1", X_brain, y_brain)
bhoa_pros, mask_pros, fit_pros, acc_pros, df_lym = rodar_bhoa_no_dataset_tabela("Prostate_Tumor", X_pros, y_pros)


=== Dataset: Lymphoma ===
Formato X: (66, 4026)
Classes: {'CLL': np.int64(11), 'DLBCL': np.int64(46), 'FL': np.int64(9)}
teste
Ajustando cv n_splits de 10 para 9 (classe mínima=9)
Melhor fitness: 0.9979
Nº de genes selecionados (após MRMR): 21 de 100
Acurácia estimada (a partir do fitness): 1.0000
Índices das features selecionadas no X original:
[2735 3753 1263  162 3746  100  767 2134  756 3596 1831 2121 3700 2949
 1260 1115 1321 2744 2778 1319 1367]

Resultado resumido:
Dataset  Accuracy (%)                    Number of Genes      Running Time (s)
             Best      Mean  Worst  STD        Mean      STD                  
Lymphoma    100.00    100.00 100.00 0.00        21       0.00     2873.37     
